# Capstone Part 1: Exploratory Data Analysis & Integrity Audit

## Student & Course Information
* **Student Name:** Mohammad Owais
* **Registration Number:** iitp_aiml_2506198
* **Email:** zkzonfamily@gmail.com
* **Course:** IIT Patna AI/ML Course Capstone
* **Project Repository:** `d2c-churn-data-audit`

---

### Objective:
This notebook implements a complete data loading, joining, cleaning, and visual analysis workflow to evaluate customer churn drivers while aggressively protecting against target window data leakage.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style='whitegrid')
os.makedirs('outputs', exist_ok=True)
print('Libraries and directory structures loaded successfully.')

Libraries and directory structures loaded successfully.


## 1. Data Loading and Base Table Inspections
We load all core data assets and verify shapes against known operational parameters.

In [2]:
customers = pd.read_csv('customers.csv')
orders = pd.read_csv('orders.csv')
support = pd.read_csv('support_tickets.csv')
labels = pd.read_csv('churn_labels.csv')
web_events = pd.read_csv('web_events_snapshot.csv')
rfm_snap = pd.read_csv('rfm_modeling_snapshot.csv')
interventions = pd.read_csv('intervention_history.csv')

print(f'Customers Profile Layer: {customers.shape}')
print(f'Raw Transactions File:   {orders.shape}')
print(f'Customer Care Tickets:    {support.shape}')
print(f'Target Churn Labels:     {labels.shape}')

Customers Profile Layer: (2400, 9)
Raw Transactions File:   (10009, 10)
Customer Care Tickets:    (1921, 8)
Target Churn Labels:     (2400, 4)


## 2. Implementing Rigorous Data Quality Cleansing
Based on our data audit, we apply exact deterministic fixes for duplicates, missing categorical strings, and split out the post-snapshot leakage transaction records.

In [3]:
# A. Deduplicate retry webhook records
orders_cleaned = orders[~orders['order_id'].str.endswith('_DUP')].copy()

# B. Categorical Missing Profiling Imputation
customers['loyalty_tier'] = customers['loyalty_tier'].fillna('UNKNOWN_UNENROLLED')
customers['skin_type'] = customers['skin_type'].fillna('UNKNOWN_PROFILE')

# C. Unrated experiences neutral transformation
orders_cleaned['rating'] = orders_cleaned['rating'].fillna(4.0)

# D. LEAKAGE ISOLATION GUARDRAIL
orders_cleaned['order_date'] = pd.to_datetime(orders_cleaned['order_date'])
historical_orders = orders_cleaned[orders_cleaned['order_date'] <= '2025-09-30'].copy()

print(f'Cleaned Historical Transact Rows (Pre-Snapshot Base): {historical_orders.shape[0]}')

Cleaned Historical Transact Rows (Pre-Snapshot Base): 8128


## 3. Creating Visual Diagnostics (+6 Required Meaningful Visual Charts)

In [4]:
# Plot 1: Target Variable Breakdown
plt.figure(figsize=(6, 4))
sns.countplot(data=labels, x='churn_label', hue='churn_label', palette='viridis', legend=False)
plt.title('Distribution of Churn Labels (Target Balance Check)')
plt.xlabel('Churn Target (1 = Churned, 0 = Retained)')
plt.savefig('outputs/chart1_target_balance.png', dpi=300, bbox_inches='tight')
plt.close()

ValueError: Could not interpret value `churn_label` for `x`. An entry with this name does not appear in `data`.

<Figure size 600x400 with 0 Axes>

In [ ]:
# Plot 2: Support Ticket Frequencies by Category
plt.figure(figsize=(8, 4))
sns.countplot(data=support, y='issue_type', order=support['issue_type'].value_counts().index, palette='magma', hue='issue_type', legend=False)
plt.title('Support Desk Issue Volume by Category Type')
plt.xlabel('Total Logged Tickets')
plt.savefig('outputs/chart2_support_categories.png', dpi=300, bbox_inches='tight')
plt.close()

In [ ]:
# Plot 3: Boxplot of Recency Days vs Churn State
merged_rfm = pd.merge(rfm_snap, labels, on='customer_id')
plt.figure(figsize=(7, 4))
sns.boxplot(data=merged_rfm, x='churn_label', y='recency_days', palette='Set2', hue='churn_label', legend=False)
plt.title('Order Recency Spans vs. Observed Customer Churn')
plt.ylabel('Days Since Most Recent Order')
plt.savefig('outputs/chart3_recency_boxplot.png', dpi=300, bbox_inches='tight')
plt.close()

In [ ]:
# Plot 4: Order Frequency Distribution per Customer
plt.figure(figsize=(7, 4))
sns.histplot(data=rfm_snap, x='frequency_180d', kde=True, bins=15, color='royalblue')
plt.title('180-Day Customer Historical Purchase Frequency')
plt.xlabel('Number of Orders Logged')
plt.savefig('outputs/chart4_frequency_distribution.png', dpi=300, bbox_inches='tight')
plt.close()

In [ ]:
# Plot 5: App Platform Sessions vs Observed Churn
plt.figure(figsize=(7, 4))
sns.violinplot(data=merged_rfm, x='churn_label', y='sessions_30d', palette='coolwarm', hue='churn_label', legend=False)
plt.title('30-Day Platform Session Frequencies vs Churn Outcomes')
plt.ylabel('Total Volumetric Sessions')
plt.savefig('outputs/chart5_session_violin.png', dpi=300, bbox_inches='tight')
plt.close()

In [ ]:
# Plot 6: Financial Monetary Distribution vs Acquisition Footprint
plt.figure(figsize=(8, 4))
sns.barplot(data=merged_rfm, x='acquisition_channel', y='monetary_180d', hue='churn_label', palette='muted', errorbar=None)
plt.title('Mean 180-Day Revenue Contributions by Sourcing Vector')
plt.ylabel('Average Gross Revenue Value (₹)')
plt.xticks(rotation=15)
plt.savefig('outputs/chart6_monetary_sourcing.png', dpi=300, bbox_inches='tight')
plt.close()
print('All 6 high-resolution analysis plots generated and cached successfully.')